In [20]:
import gurobipy as gp
from gurobipy import GRB

In [18]:
lpex = gp.Model("lpex")

x1 = lpex.addVar()
x2 = lpex.addVar(vtype=GRB.CONTINUOUS, lb=0.0)
x3 = lpex.addVar()
x4 = lpex.addVar()

In [5]:
lpex.setObjective(9*x1 + 5*x2 + 4*x3 + x4, GRB.MAXIMIZE)

In [6]:
lpex.addConstr(2*x1 + x2 + x3 + 2*x4 <= 2)
lpex.addConstr(8*x1 + 4*x2 -2*x3 - x4 >= 10)
lpex.addConstr(4*x1 + 7*x2 + 2*x3 + x4 <= 4)

<gurobi.Constr *Awaiting Model Update*>

In [7]:
lpex.optimize()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 3 rows, 4 columns and 12 nonzeros (Max)
Model fingerprint: 0x95c291aa
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [1e+00, 9e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 1e+01]

Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible or unbounded model


In [8]:
lpex.params.DualReductions = 0
lpex.optimize()

Set parameter DualReductions to value 0
Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
DualReductions  0

Optimize a model with 3 rows, 4 columns and 12 nonzeros (Max)
Model fingerprint: 0x95c291aa
Model has 4 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 8e+00]
  Objective range  [1e+00, 9e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [2e+00, 1e+01]

Presolve time: 0.01s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Infeasible model


In [9]:
categories, minNutrition, maxNutrition = gp.multidict(
    {
        "calories": [1800, 2200],
        "protein": [91, GRB.INFINITY],
        "fat": [0, 65],
        "sodium": [0, 1779],
    }
)

foods, cost = gp.multidict(
    {
        "hamburger": 2.49,
        "chicken": 2.89,
        "hot dog": 1.50,
        "fries": 1.89,
        "macaroni": 2.09,
        "pizza": 1.99,
        "salad": 2.49,
        "milk": 0.89,
        "ice cream": 1.59,
    }
)

In [14]:
nutritionValues = {
    ("hamburger", "calories"): 410,
    ("hamburger", "protein"): 24,
    ("hamburger", "fat"): 26,
    ("hamburger", "sodium"): 730,
    ("chicken", "calories"): 420,
    ("chicken", "protein"): 32,
    ("chicken", "fat"): 10,
    ("chicken", "sodium"): 1190,
    ("hot dog", "calories"): 560,
    ("hot dog", "protein"): 20,
    ("hot dog", "fat"): 32,
    ("hot dog", "sodium"): 1800,
    ("fries", "calories"): 380,
    ("fries", "protein"): 4,
    ("fries", "fat"): 19,
    ("fries", "sodium"): 270,
    ("macaroni", "calories"): 320,
    ("macaroni", "protein"): 12,
    ("macaroni", "fat"): 10,
    ("macaroni", "sodium"): 930,
    ("pizza", "calories"): 320,
    ("pizza", "protein"): 15,
    ("pizza", "fat"): 12,
    ("pizza", "sodium"): 820,
    ("salad", "calories"): 320,
    ("salad", "protein"): 31,
    ("salad", "fat"): 12,
    ("salad", "sodium"): 1230,
    ("milk", "calories"): 100,
    ("milk", "protein"): 8,
    ("milk", "fat"): 2.5,
    ("milk", "sodium"): 125,
    ("ice cream", "calories"): 330,
    ("ice cream", "protein"): 8,
    ("ice cream", "fat"): 10,
    ("ice cream", "sodium"): 180,
}

In [25]:
m = gp.Model("diet")
buy = m.addVars(foods, name="buy")
m.setObjective(buy.prod(cost), GRB.MINIMIZE)

In [26]:
for c in categories:
    m.addRange(sum(nutritionValues[f, c] * buy[f] for f in foods), minNutrition[c], maxNutrition[c], c)

In [34]:
def printSolution():
    if m.status == GRB.OPTIMAL:
        print(f"\nCost: {m.ObjVal:g}")
        print("\nBuy:")
        for f in foods:
            if buy[f].X > 0.0001:
                print(f"{f} {buy[f].X:g}")
    else:
        print("No solution")

In [35]:
m.optimize()
printSolution()

Gurobi Optimizer version 13.0.1 build v13.0.1rc0 (mac64[arm] - Darwin 24.5.0 24F74)

CPU model: Apple M2 Pro
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Optimize a model with 4 rows, 12 columns and 39 nonzeros (Min)
Model has 9 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e+00, 2e+03]
  Objective range  [9e-01, 3e+00]
  Bounds range     [6e+01, 2e+03]
  RHS range        [6e+01, 2e+03]


Solved in 0 iterations and 0.00 seconds (0.00 work units)
Optimal objective  1.182886111e+01

Cost: 11.8289

Buy:
hamburger 0.604514
milk 6.97014
ice cream 2.59132
